In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — LOAD ROUND 5 DATA
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROUND = 5
DAYS = [2, 3, 4]

DATA_DIR = Path("Data/round5")

ALGO_PRODUCTS = [
    # Galaxy Sounds Recorders
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",

    # Vertical Sleeping Pods
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",

    # Organic Microchips
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",

    # Purification Pebbles
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",

    # Domestic Robots
    "ROBOT_VACUUMING",
    "ROBOT_MOPPING",
    "ROBOT_DISHES",
    "ROBOT_LAUNDRY",
    "ROBOT_IRONING",

    # UV-Visors
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",

    # Instant Translators
    "TRANSLATOR_SPACE_GRAY",
    "TRANSLATOR_ASTRO_BLACK",
    "TRANSLATOR_ECLIPSE_CHARCOAL",
    "TRANSLATOR_GRAPHITE_MIST",
    "TRANSLATOR_VOID_BLUE",

    # Construction Panels
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",

    # Liquid Breath Oxygen Shakes
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",

    # Protein Snack Packs
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]

POSITION_LIMITS = {product: 10 for product in ALGO_PRODUCTS}

prices_parts = []
trades_parts = []

for day in DAYS:
    price_path = DATA_DIR / f"prices_round_{ROUND}_day_{day}.csv"
    trade_path = DATA_DIR / f"trades_round_{ROUND}_day_{day}.csv"

    p = pd.read_csv(price_path, sep=";")
    t = pd.read_csv(trade_path, sep=";")

    p["file_day"] = day
    t["file_day"] = day

    prices_parts.append(p)
    trades_parts.append(t)

prices = pd.concat(prices_parts, ignore_index=True)
trades = pd.concat(trades_parts, ignore_index=True)

# Standardise trade product column name.
if "symbol" in trades.columns and "product" not in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Keep only valid Round 5 algorithmic products.
prices = prices[prices["product"].isin(ALGO_PRODUCTS)].copy()
trades = trades[trades["product"].isin(ALGO_PRODUCTS)].copy()

# Useful global time index across days.
# Assumes timestamp resets each day.
min_day = min(DAYS)
prices["global_ts"] = (prices["file_day"] - min_day) * 1_000_000 + prices["timestamp"]
trades["global_ts"] = (trades["file_day"] - min_day) * 1_000_000 + trades["timestamp"]

prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)
trades = trades.sort_values(["product", "global_ts"]).reset_index(drop=True)

# Basic sanity checks.
price_products = sorted(prices["product"].unique())
trade_products = sorted(trades["product"].unique())

missing_in_prices = sorted(set(ALGO_PRODUCTS) - set(price_products))
missing_in_trades = sorted(set(ALGO_PRODUCTS) - set(trade_products))

print("prices shape:", prices.shape)
print("trades shape:", trades.shape)
print()
print("Price products:", price_products)
print("Trade products:", trade_products)
print()
print("Missing in prices:", missing_in_prices)
print("Missing in trades:", missing_in_trades)
print()
print("Price days:", sorted(prices["file_day"].unique()))
print("Trade days:", sorted(trades["file_day"].unique()))
print()
print("Position limits:", POSITION_LIMITS)

display(prices.head())
display(trades.head())

prices shape: (1500000, 19)
trades shape: (35385, 9)

Price products: ['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_EVENING_BREATH', 'OXYGEN_SHAKE_GARLIC', 'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_MORNING_BREATH', 'PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4', 'PEBBLES_L', 'PEBBLES_M', 'PEBBLES_S', 'PEBBLES_XL', 'PEBBLES_XS', 'ROBOT_DISHES', 'ROBOT_IRONING', 'ROBOT_LAUNDRY', 'ROBOT_MOPPING', 'ROBOT_VACUUMING', 'SLEEP_POD_COTTON', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_NYLON', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_SUEDE', 'SNACKPACK_CHOCOLATE', 'SNACKPACK_PISTACHIO', 'SNACKPACK_RASPBERRY', 'SNACKPACK_STRAWBERRY', 'SNACKPACK_VANILLA', 'TRANSLATOR_ASTRO_BLACK', 'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST', 'TRANSLATOR_SPACE_GRAY'

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,file_day,global_ts
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,2,0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,NaN,10014,18,10016.0,25.0,NaN,NaN,10007.5,0.0,2,100
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,NaN,10009,19,10011.0,31.0,NaN,NaN,10002.5,0.0,2,200
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,NaN,10007,25,10009.0,33.0,NaN,NaN,10000.5,0.0,2,300
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,NaN,10012,14,10013.0,32.0,NaN,NaN,10005.5,0.0,2,400


,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts
0,1700,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9969.0,4,2,1700
1,14500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9749.0,1,2,14500
2,15100,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9764.0,2,2,15100
3,26500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9656.0,4,2,26500
4,36400,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9675.0,4,2,36400


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — CATEGORY DEFINITIONS + RESEARCH CONFIG
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
from itertools import combinations, product as iter_product
import math
import warnings

warnings.filterwarnings("ignore")

OUT_DIR = Path("outputs_deep_scan")
OUT_DIR.mkdir(exist_ok=True)

CATEGORIES = {
    "Galaxy Sounds": [
        "GALAXY_SOUNDS_DARK_MATTER",
        "GALAXY_SOUNDS_BLACK_HOLES",
        "GALAXY_SOUNDS_PLANETARY_RINGS",
        "GALAXY_SOUNDS_SOLAR_WINDS",
        "GALAXY_SOUNDS_SOLAR_FLAMES",
    ],
    "Sleeping Pods": [
        "SLEEP_POD_SUEDE",
        "SLEEP_POD_LAMB_WOOL",
        "SLEEP_POD_POLYESTER",
        "SLEEP_POD_NYLON",
        "SLEEP_POD_COTTON",
    ],
    "Organic Microchips": [
        "MICROCHIP_CIRCLE",
        "MICROCHIP_OVAL",
        "MICROCHIP_SQUARE",
        "MICROCHIP_RECTANGLE",
        "MICROCHIP_TRIANGLE",
    ],
    "Pebbles": [
        "PEBBLES_XS",
        "PEBBLES_S",
        "PEBBLES_M",
        "PEBBLES_L",
        "PEBBLES_XL",
    ],
    "Robots": [
        "ROBOT_VACUUMING",
        "ROBOT_MOPPING",
        "ROBOT_DISHES",
        "ROBOT_LAUNDRY",
        "ROBOT_IRONING",
    ],
    "UV Visors": [
        "UV_VISOR_YELLOW",
        "UV_VISOR_AMBER",
        "UV_VISOR_ORANGE",
        "UV_VISOR_RED",
        "UV_VISOR_MAGENTA",
    ],
    "Translators": [
        "TRANSLATOR_SPACE_GRAY",
        "TRANSLATOR_ASTRO_BLACK",
        "TRANSLATOR_ECLIPSE_CHARCOAL",
        "TRANSLATOR_GRAPHITE_MIST",
        "TRANSLATOR_VOID_BLUE",
    ],
    "Panels": [
        "PANEL_1X2",
        "PANEL_2X2",
        "PANEL_1X4",
        "PANEL_2X4",
        "PANEL_4X4",
    ],
    "Oxygen Shakes": [
        "OXYGEN_SHAKE_MORNING_BREATH",
        "OXYGEN_SHAKE_EVENING_BREATH",
        "OXYGEN_SHAKE_MINT",
        "OXYGEN_SHAKE_CHOCOLATE",
        "OXYGEN_SHAKE_GARLIC",
    ],
    "Snackpacks": [
        "SNACKPACK_CHOCOLATE",
        "SNACKPACK_VANILLA",
        "SNACKPACK_PISTACHIO",
        "SNACKPACK_STRAWBERRY",
        "SNACKPACK_RASPBERRY",
    ],
}

# Broad scan settings.
PAST_HORIZONS = [1, 2, 5, 10, 25, 50, 100, 250]
FUTURE_HORIZONS = [1, 2, 5, 10, 25, 50, 100, 250]
ROLLING_WINDOWS = [100, 250, 500, 1000, 2500]
FLOW_WINDOWS = [500, 1000, 2500, 5000, 10000]

# Exhaustive but still manageable.
TOP_N_PER_FAMILY_PER_CATEGORY = 20
TOP_N_BACKTEST_PER_CATEGORY = 25

print("Loaded categories:", list(CATEGORIES.keys()))

Loaded categories: ['Galaxy Sounds', 'Sleeping Pods', 'Organic Microchips', 'Pebbles', 'Robots', 'UV Visors', 'Translators', 'Panels', 'Oxygen Shakes', 'Snackpacks']


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — GLOBAL FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════════════

def add_book_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["mid"] = df["mid_price"]
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]

    for col in [
        "bid_volume_1", "bid_volume_2", "bid_volume_3",
        "ask_volume_1", "ask_volume_2", "ask_volume_3",
    ]:
        df[col] = df[col].fillna(0)

    df["bid_depth_l1"] = df["bid_volume_1"]
    df["ask_depth_l1"] = df["ask_volume_1"]

    df["bid_depth_total"] = df["bid_volume_1"] + df["bid_volume_2"] + df["bid_volume_3"]
    df["ask_depth_total"] = df["ask_volume_1"] + df["ask_volume_2"] + df["ask_volume_3"]
    df["depth_total"] = df["bid_depth_total"] + df["ask_depth_total"]

    df["imbalance_l1"] = (
        (df["bid_depth_l1"] - df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["imbalance_total"] = (
        (df["bid_depth_total"] - df["ask_depth_total"])
        / (df["bid_depth_total"] + df["ask_depth_total"]).replace(0, np.nan)
    )

    df["microprice_l1"] = (
        (df["ask_price_1"] * df["bid_depth_l1"] + df["bid_price_1"] * df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["microprice_edge"] = df["microprice_l1"] - df["mid"]
    df["microprice_edge_norm"] = df["microprice_edge"] / df["spread"].replace(0, np.nan)

    return df


prices_f = add_book_features(prices)

print("prices_f:", prices_f.shape)
display(prices_f.head())

prices_f: (1500000, 31)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,bid_depth_l1,ask_depth_l1,bid_depth_total,ask_depth_total,depth_total,imbalance_l1,imbalance_total,microprice_l1,microprice_edge,microprice_edge_norm
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,0.0,10006,...,22,22,48.0,48.0,96.0,0.0,0.0,10000.0,0.0,0.0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,0.0,10014,...,18,18,43.0,43.0,86.0,0.0,0.0,10007.5,0.0,0.0
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,0.0,10009,...,19,19,50.0,50.0,100.0,0.0,0.0,10002.5,0.0,0.0
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,0.0,10007,...,25,25,58.0,58.0,116.0,0.0,0.0,10000.5,0.0,0.0
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,0.0,10012,...,14,14,46.0,46.0,92.0,0.0,0.0,10005.5,0.0,0.0


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — UTILITY FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def clean_name(s: str) -> str:
    return s.lower().replace(" ", "_").replace("/", "_").replace("-", "_")


def safe_corr(x, y):
    valid = x.notna() & y.notna()
    if valid.sum() < 200:
        return np.nan

    xv = x[valid]
    yv = y[valid]

    if xv.std() == 0 or yv.std() == 0:
        return np.nan

    return xv.corr(yv)


def directional_hit(signal, future_move):
    valid = signal.notna() & future_move.notna()
    if valid.sum() < 200:
        return np.nan

    x = signal[valid]
    y = future_move[valid]

    nonzero = (x != 0) & (y != 0)
    if nonzero.sum() < 50:
        return np.nan

    return (np.sign(x[nonzero]) == np.sign(y[nonzero])).mean()


def directional_edge(signal, future_move):
    valid = signal.notna() & future_move.notna()
    if valid.sum() < 200:
        return np.nan

    x = signal[valid]
    y = future_move[valid]

    s = np.sign(x)
    active = s != 0

    if active.sum() < 50:
        return np.nan

    return (s[active] * y[active]).mean()


def zscore_series(s, window=500, min_periods=100):
    mean = s.rolling(window, min_periods=min_periods).mean()
    std = s.rolling(window, min_periods=min_periods).std()
    return (s - mean) / std.replace(0, np.nan)


def fit_ols_y_on_x(y: pd.Series, x: pd.Series):
    valid = y.notna() & x.notna()
    yv = y[valid].values
    xv = x[valid].values

    if len(yv) < 200 or np.var(xv) == 0:
        return np.nan, np.nan

    beta = np.cov(yv, xv, ddof=0)[0, 1] / np.var(xv)
    alpha = np.mean(yv) - beta * np.mean(xv)
    return alpha, beta


def mean_reversion_slope_and_halflife(s: pd.Series):
    s = s.dropna()

    if len(s) < 300:
        return np.nan, np.nan, np.nan

    lagged = s.shift(1)
    delta = s.diff()
    valid = lagged.notna() & delta.notna()

    x = lagged[valid].values
    y = delta[valid].values

    if len(x) < 300 or np.var(x) == 0:
        return s.autocorr(1), np.nan, np.nan

    slope = np.cov(y, x, ddof=0)[0, 1] / np.var(x)

    if slope < 0:
        half_life = -np.log(2) / slope
    else:
        half_life = np.inf

    return s.autocorr(1), slope, half_life


def get_category_data(category_name):
    products = CATEGORIES[category_name]

    cp = (
        prices_f[prices_f["product"].isin(products)]
        .sort_values(["file_day", "timestamp", "product"])
        .reset_index(drop=True)
    )

    ct = (
        trades[trades["product"].isin(products)]
        .sort_values(["file_day", "timestamp", "product"])
        .reset_index(drop=True)
    )

    def make_wide(value_col):
        wide = (
            cp.pivot_table(
                index=["file_day", "timestamp"],
                columns="product",
                values=value_col,
                aggfunc="last",
            )
            .sort_index()
            .reindex(columns=products)
        )
        return wide

    wide = {
        "mid": make_wide("mid"),
        "bid1": make_wide("bid_price_1"),
        "ask1": make_wide("ask_price_1"),
        "bidvol1": make_wide("bid_volume_1"),
        "askvol1": make_wide("ask_volume_1"),
        "spread": make_wide("spread"),
        "imbalance_l1": make_wide("imbalance_l1"),
        "imbalance_total": make_wide("imbalance_total"),
        "microprice_edge": make_wide("microprice_edge"),
        "microprice_edge_norm": make_wide("microprice_edge_norm"),
    }

    return products, cp, ct, wide

In [5]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — BROAD STRATEGY-FAMILY SCANNERS
# ════════════════════════════════════════════════════════════════════════════

def scan_product_diagnostics(category_name, products, cp, ct):
    diag = (
        cp.groupby(["file_day", "product"])
        .agg(
            rows=("timestamp", "count"),
            first_mid=("mid", "first"),
            last_mid=("mid", "last"),
            min_mid=("mid", "min"),
            max_mid=("mid", "max"),
            mid_range=("mid", lambda x: x.max() - x.min()),
            mean_spread=("spread", "mean"),
            median_spread=("spread", "median"),
            mean_depth=("depth_total", "mean"),
            std_imb_l1=("imbalance_l1", "std"),
        )
        .reset_index()
    )

    trade_diag = (
        ct.groupby(["file_day", "product"])
        .agg(
            trades=("price", "count"),
            total_qty=("quantity", "sum"),
            mean_trade_price=("price", "mean"),
        )
        .reset_index()
    )

    diag["category"] = category_name
    trade_diag["category"] = category_name

    return diag, trade_diag


def scan_return_correlations(category_name, products, wide):
    mid = wide["mid"]
    rows = []

    for h in FUTURE_HORIZONS:
        ret = mid.groupby(level=0).diff(h)

        for a, b in combinations(products, 2):
            c = ret[a].corr(ret[b])

            rows.append({
                "category": category_name,
                "horizon": h,
                "a": a,
                "b": b,
                "return_corr": c,
                "abs_return_corr": abs(c) if pd.notna(c) else np.nan,
            })

    return pd.DataFrame(rows).sort_values("abs_return_corr", ascending=False)


def scan_basket_invariants(category_name, products, wide):
    """
    Tests:
    1. Equal-weight sum.
    2. Small integer linear combinations, including all-positive sums.
    This is designed to rediscover Pebbles-like invariants and other hidden formula relationships.
    """
    mid = wide["mid"]
    rows = []

    # Equal-weight basket.
    for day in DAYS:
        dm = mid.loc[day]
        combo = dm.sum(axis=1)

        rows.append({
            "category": category_name,
            "day": day,
            "type": "equal_sum",
            "coefs": str([1] * len(products)),
            "combo_mean": combo.mean(),
            "combo_std": combo.std(),
            "combo_range": combo.max() - combo.min(),
            "combo_min": combo.min(),
            "combo_max": combo.max(),
        })

    # Integer coefficients. This is heavier but useful.
    coef_values = [-2, -1, 0, 1, 2]
    seen = set()

    for coefs in iter_product(coef_values, repeat=len(products)):
        if all(c == 0 for c in coefs):
            continue

        # Skip very sparse single-product combos.
        if sum(c != 0 for c in coefs) < 2:
            continue

        # Normalise sign and gcd to avoid duplicates.
        coef_list = list(coefs)
        gcd = 0
        for c in coef_list:
            gcd = math.gcd(gcd, abs(c))
        if gcd > 1:
            coef_list = [c // gcd for c in coef_list]

        first_nonzero = next(c for c in coef_list if c != 0)
        if first_nonzero < 0:
            coef_list = [-c for c in coef_list]

        key = tuple(coef_list)
        if key in seen:
            continue
        seen.add(key)

        # Avoid extremely weird all-same duplicates except equal sum already included.
        if key == tuple([1] * len(products)):
            continue

        for day in DAYS:
            dm = mid.loc[day]
            mat = dm.values
            coef_arr = np.array(coef_list)
            combo = pd.Series(mat @ coef_arr, index=dm.index)

            rows.append({
                "category": category_name,
                "day": day,
                "type": "integer_combo",
                "coefs": str(coef_list),
                "combo_mean": combo.mean(),
                "combo_std": combo.std(),
                "combo_range": combo.max() - combo.min(),
                "combo_min": combo.min(),
                "combo_max": combo.max(),
            })

    raw = pd.DataFrame(rows)

    agg = (
        raw.groupby(["category", "type", "coefs"])
        .agg(
            days=("day", "count"),
            avg_std=("combo_std", "mean"),
            max_std=("combo_std", "max"),
            avg_range=("combo_range", "mean"),
            max_range=("combo_range", "max"),
            avg_abs_mean=("combo_mean", lambda x: np.mean(np.abs(x))),
        )
        .reset_index()
    )

    # Lower range/std is interesting. For large mean baskets, range/mean is also informative.
    agg["range_to_mean"] = agg["avg_range"] / agg["avg_abs_mean"].replace(0, np.nan)
    agg = agg.sort_values(["range_to_mean", "avg_range"], ascending=True)

    return raw, agg


def scan_pair_spreads(category_name, products, wide):
    mid = wide["mid"]
    rows = []

    for day in DAYS:
        dm = mid.loc[day]

        for a, b in combinations(products, 2):
            alpha, beta = fit_ols_y_on_x(dm[a], dm[b])
            spread = dm[a] - (alpha + beta * dm[b])

            lag1, slope, half_life = mean_reversion_slope_and_halflife(spread)

            for w in ROLLING_WINDOWS:
                z = zscore_series(spread, window=w, min_periods=max(50, w // 5))

                rows.append({
                    "category": category_name,
                    "day": day,
                    "a": a,
                    "b": b,
                    "window": w,
                    "alpha": alpha,
                    "beta": beta,
                    "spread_std": spread.std(),
                    "spread_range": spread.max() - spread.min(),
                    "lag1": lag1,
                    "mr_slope": slope,
                    "half_life": half_life,
                    "z_abs_gt_2": int((z.abs() > 2).sum()),
                    "z_abs_gt_3": int((z.abs() > 3).sum()),
                })

    raw = pd.DataFrame(rows)

    agg = (
        raw.groupby(["category", "a", "b", "window"])
        .agg(
            days=("day", "count"),
            avg_beta=("beta", "mean"),
            beta_std=("beta", "std"),
            avg_spread_std=("spread_std", "mean"),
            avg_spread_range=("spread_range", "mean"),
            avg_lag1=("lag1", "mean"),
            avg_half_life=("half_life", "mean"),
            total_z_abs_gt_2=("z_abs_gt_2", "sum"),
            total_z_abs_gt_3=("z_abs_gt_3", "sum"),
        )
        .reset_index()
    )

    agg["finite_half_life"] = np.isfinite(agg["avg_half_life"])
    agg["pair_score"] = (
        agg["total_z_abs_gt_2"]
        / (1 + agg["avg_spread_std"].abs())
        * agg["finite_half_life"].map({True: 1.0, False: 0.25})
    )

    agg = agg.sort_values("pair_score", ascending=False)

    return raw, agg


def scan_pca_residuals(category_name, products, wide):
    mid = wide["mid"]
    rows = []
    explained_rows = []
    loading_rows = []

    for day in DAYS:
        dm = mid.loc[day]
        x = dm - dm.iloc[0]

        xz = (x - x.mean()) / x.std().replace(0, np.nan)
        xz = xz.fillna(0)

        U, S, Vt = np.linalg.svd(xz.values, full_matrices=False)
        explained = (S ** 2) / np.sum(S ** 2)

        explained_rows.append({
            "category": category_name,
            "day": day,
            **{f"pc{i+1}": explained[i] for i in range(min(5, len(explained)))}
        })

        for product_name, loading in zip(products, Vt[0]):
            loading_rows.append({
                "category": category_name,
                "day": day,
                "product": product_name,
                "pc1_loading": loading,
            })

        pc1_scores = U[:, [0]] * S[0]
        pc1_recon = pc1_scores @ Vt[[0], :]

        resid = pd.DataFrame(
            xz.values - pc1_recon,
            index=xz.index,
            columns=xz.columns,
        )

        for p in products:
            r = resid[p]

            lag1, slope, half_life = mean_reversion_slope_and_halflife(r)

            for w in ROLLING_WINDOWS:
                z = zscore_series(r, window=w, min_periods=max(50, w // 5))

                rows.append({
                    "category": category_name,
                    "day": day,
                    "product": p,
                    "window": w,
                    "resid_std": r.std(),
                    "resid_range": r.max() - r.min(),
                    "lag1": lag1,
                    "mr_slope": slope,
                    "half_life": half_life,
                    "z_abs_gt_2": int((z.abs() > 2).sum()),
                    "z_abs_gt_3": int((z.abs() > 3).sum()),
                })

    raw = pd.DataFrame(rows)
    explained_df = pd.DataFrame(explained_rows)
    loadings_df = pd.DataFrame(loading_rows)

    agg = (
        raw.groupby(["category", "product", "window"])
        .agg(
            days=("day", "count"),
            avg_resid_std=("resid_std", "mean"),
            avg_resid_range=("resid_range", "mean"),
            avg_lag1=("lag1", "mean"),
            avg_half_life=("half_life", "mean"),
            total_z_abs_gt_2=("z_abs_gt_2", "sum"),
            total_z_abs_gt_3=("z_abs_gt_3", "sum"),
        )
        .reset_index()
    )

    pc1_strength = explained_df.groupby("category")["pc1"].mean().to_dict()
    agg["avg_pc1_explained"] = agg["category"].map(pc1_strength)

    agg["pca_score"] = (
        agg["total_z_abs_gt_2"]
        * agg["avg_pc1_explained"]
        / (1 + agg["avg_resid_std"])
    )

    agg = agg.sort_values("pca_score", ascending=False)

    return raw, agg, explained_df, loadings_df


def attach_trade_flow(cp, ct):
    quote_cols = [
        "file_day", "timestamp", "product",
        "bid_price_1", "ask_price_1", "mid", "spread",
    ]

    tq = ct.merge(
        cp[quote_cols],
        on=["file_day", "timestamp", "product"],
        how="left",
    )

    tq["inferred_side"] = 0
    tq.loc[tq["price"] >= tq["ask_price_1"], "inferred_side"] = 1
    tq.loc[tq["price"] <= tq["bid_price_1"], "inferred_side"] = -1
    tq["signed_qty"] = tq["inferred_side"] * tq["quantity"]

    return tq


def make_product_day_signal_frame(cp, ct, product_name, day):
    p = (
        cp[(cp["product"] == product_name) & (cp["file_day"] == day)]
        .sort_values("timestamp")
        .copy()
        .reset_index(drop=True)
    )

    if len(p) == 0:
        return pd.DataFrame()

    tq = attach_trade_flow(cp, ct)
    tq = tq[(tq["product"] == product_name) & (tq["file_day"] == day)]

    flow = (
        tq.groupby("timestamp")
        .agg(
            signed_qty=("signed_qty", "sum"),
            gross_qty=("quantity", "sum"),
        )
        .reindex(p["timestamp"])
        .fillna(0)
        .reset_index(drop=True)
    )

    p["signed_qty"] = flow["signed_qty"]
    p["gross_qty"] = flow["gross_qty"]

    signals = pd.DataFrame(index=p.index)
    signals["mid"] = p["mid"]

    # Order book signals
    signals["book_imb_l1"] = p["imbalance_l1"]
    signals["book_imb_total"] = p["imbalance_total"]
    signals["microprice_edge"] = p["microprice_edge"]
    signals["microprice_edge_norm"] = p["microprice_edge_norm"]

    # Momentum / mean-reversion over past move
    for h in PAST_HORIZONS:
        past_move = p["mid"].diff(h)
        signals[f"mom_ret_{h}"] = past_move
        signals[f"rev_ret_{h}"] = -past_move

    # Rolling z-score mean reversion / breakout
    for w in ROLLING_WINDOWS:
        z = zscore_series(p["mid"], window=w, min_periods=max(50, w // 5))
        signals[f"meanrev_z_{w}"] = -z
        signals[f"breakout_z_{w}"] = z

    # Spread regime
    for w in ROLLING_WINDOWS:
        spread_z = zscore_series(p["spread"], window=w, min_periods=max(50, w // 5))
        signals[f"spread_z_{w}"] = spread_z

    # Trade-flow signals
    for fw in FLOW_WINDOWS:
        grid_w = max(1, fw // 100)
        signed_roll = p["signed_qty"].rolling(grid_w, min_periods=1).sum()
        gross_roll = p["gross_qty"].rolling(grid_w, min_periods=1).sum()
        flow_imb = signed_roll / gross_roll.replace(0, np.nan)

        signals[f"flow_imb_{fw}"] = flow_imb.fillna(0)

    return p, signals


def scan_single_product_signals(category_name, products, cp, ct):
    rows = []

    for day in DAYS:
        for p_name in products:
            p, sigs = make_product_day_signal_frame(cp, ct, p_name, day)

            if len(p) == 0:
                continue

            for sig_name in sigs.columns:
                if sig_name == "mid":
                    continue

                signal = sigs[sig_name]

                for fh in FUTURE_HORIZONS:
                    future_move = p["mid"].shift(-fh) - p["mid"]

                    corr = safe_corr(signal, future_move)
                    hit = directional_hit(signal, future_move)
                    edge = directional_edge(signal, future_move)

                    rows.append({
                        "category": category_name,
                        "day": day,
                        "product": p_name,
                        "signal": sig_name,
                        "future_h": fh,
                        "corr": corr,
                        "hit_rate": hit,
                        "directional_edge": edge,
                        "n": int((signal.notna() & future_move.notna()).sum()),
                    })

    raw = pd.DataFrame(rows)

    agg = (
        raw.groupby(["category", "product", "signal", "future_h"])
        .agg(
            days=("day", "count"),
            avg_corr=("corr", "mean"),
            min_corr=("corr", "min"),
            max_corr=("corr", "max"),
            avg_hit=("hit_rate", "mean"),
            avg_edge=("directional_edge", "mean"),
            min_edge=("directional_edge", "min"),
            max_edge=("directional_edge", "max"),
            avg_n=("n", "mean"),
        )
        .reset_index()
    )

    agg["same_positive_corr"] = agg["min_corr"] > 0
    agg["same_negative_corr"] = agg["max_corr"] < 0
    agg["same_positive_edge"] = agg["min_edge"] > 0
    agg["same_negative_edge"] = agg["max_edge"] < 0
    agg["abs_avg_corr"] = agg["avg_corr"].abs()

    # Consistency matters more than one huge day.
    agg["single_score"] = (
        agg["abs_avg_corr"].fillna(0)
        * 100
        + (agg["avg_hit"].fillna(0.5) - 0.5).abs() * 50
        + agg["avg_edge"].abs().fillna(0)
    )

    # Boost signals that are same-sign across all days.
    agg.loc[agg["same_positive_corr"] | agg["same_negative_corr"], "single_score"] *= 1.5
    agg.loc[agg["same_positive_edge"] | agg["same_negative_edge"], "single_score"] *= 1.5

    agg = agg.sort_values("single_score", ascending=False)

    return raw, agg


def scan_cross_product_leadlag(category_name, products, wide):
    mid = wide["mid"]
    rows = []

    for day in DAYS:
        dm = mid.loc[day]

        for leader in products:
            for follower in products:
                if leader == follower:
                    continue

                for ph in PAST_HORIZONS:
                    leader_move = dm[leader].diff(ph)

                    for fh in FUTURE_HORIZONS:
                        follower_future = dm[follower].shift(-fh) - dm[follower]

                        corr = safe_corr(leader_move, follower_future)
                        hit = directional_hit(leader_move, follower_future)
                        edge = directional_edge(leader_move, follower_future)

                        rows.append({
                            "category": category_name,
                            "day": day,
                            "leader": leader,
                            "follower": follower,
                            "past_h": ph,
                            "future_h": fh,
                            "corr": corr,
                            "hit_rate": hit,
                            "directional_edge": edge,
                        })

    raw = pd.DataFrame(rows)

    agg = (
        raw.groupby(["category", "leader", "follower", "past_h", "future_h"])
        .agg(
            days=("day", "count"),
            avg_corr=("corr", "mean"),
            min_corr=("corr", "min"),
            max_corr=("corr", "max"),
            avg_hit=("hit_rate", "mean"),
            avg_edge=("directional_edge", "mean"),
            min_edge=("directional_edge", "min"),
            max_edge=("directional_edge", "max"),
        )
        .reset_index()
    )

    agg["same_positive_corr"] = agg["min_corr"] > 0
    agg["same_negative_corr"] = agg["max_corr"] < 0
    agg["same_positive_edge"] = agg["min_edge"] > 0
    agg["same_negative_edge"] = agg["max_edge"] < 0
    agg["abs_avg_corr"] = agg["avg_corr"].abs()

    agg["leadlag_score"] = (
        agg["abs_avg_corr"].fillna(0) * 100
        + (agg["avg_hit"].fillna(0.5) - 0.5).abs() * 50
        + agg["avg_edge"].abs().fillna(0)
    )

    agg.loc[agg["same_positive_corr"] | agg["same_negative_corr"], "leadlag_score"] *= 1.5
    agg.loc[agg["same_positive_edge"] | agg["same_negative_edge"], "leadlag_score"] *= 1.5

    agg = agg.sort_values("leadlag_score", ascending=False)

    return raw, agg

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — EXECUTABLE SINGLE-PRODUCT BACKTESTER
# ════════════════════════════════════════════════════════════════════════════

def build_signal_for_candidate(cp, ct, product_name, day, signal_name):
    p, sigs = make_product_day_signal_frame(cp, ct, product_name, day)

    if signal_name not in sigs.columns:
        return p, pd.Series(0, index=p.index)

    signal = sigs[signal_name].copy()
    return p, signal


def standardize_for_trading(signal, window=500):
    z = zscore_series(signal, window=window, min_periods=max(50, window // 5))
    return z.fillna(0)


def backtest_single_product_candidate(
    cp,
    ct,
    product_name,
    signal_name,
    entry_z=1.0,
    exit_z=0.0,
    pos_limit=10,
    invert=False,
    standardize=True,
):
    day_rows = []

    for day in DAYS:
        p, signal = build_signal_for_candidate(cp, ct, product_name, day, signal_name)

        if len(p) == 0:
            continue

        if standardize:
            trad_sig = standardize_for_trading(signal, window=500)
        else:
            trad_sig = signal.fillna(0)

        if invert:
            trad_sig = -trad_sig

        # Shift to avoid same-tick lookahead.
        trad_sig = trad_sig.shift(1).fillna(0).reset_index(drop=True)

        cash = 0.0
        pos = 0
        turnover = 0
        fills = 0
        max_abs_pos = 0
        pnl_path = []

        for i, row in p.iterrows():
            s = trad_sig.iloc[i]

            target = pos

            if pos == 0:
                if s > entry_z:
                    target = pos_limit
                elif s < -entry_z:
                    target = -pos_limit
            elif pos > 0:
                if s < exit_z:
                    target = 0
            elif pos < 0:
                if s > -exit_z:
                    target = 0

            delta = target - pos

            if delta > 0:
                qty = min(delta, int(row["ask_volume_1"]))
                if qty > 0:
                    cash -= qty * row["ask_price_1"]
                    pos += qty
                    turnover += qty
                    fills += 1

            elif delta < 0:
                qty = min(-delta, int(row["bid_volume_1"]))
                if qty > 0:
                    cash += qty * row["bid_price_1"]
                    pos -= qty
                    turnover += qty
                    fills += 1

            max_abs_pos = max(max_abs_pos, abs(pos))
            pnl_path.append(cash + pos * row["mid"])

        final = p.iloc[-1]

        marked_pnl = cash + pos * final["mid"]

        liquidated = cash
        if pos > 0:
            liquidated += pos * final["bid_price_1"]
        elif pos < 0:
            liquidated -= (-pos) * final["ask_price_1"]

        day_rows.append({
            "day": day,
            "product": product_name,
            "signal": signal_name,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "invert": invert,
            "marked_pnl": marked_pnl,
            "liquidated_pnl": liquidated,
            "final_pos": pos,
            "turnover": turnover,
            "fills": fills,
            "max_abs_pos": max_abs_pos,
            "max_pnl": max(pnl_path) if pnl_path else 0,
            "min_pnl": min(pnl_path) if pnl_path else 0,
        })

    day_df = pd.DataFrame(day_rows)

    if len(day_df) == 0:
        return None, None

    summary = {
        "product": product_name,
        "signal": signal_name,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "invert": invert,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df


def backtest_top_single_candidates(category_name, cp, ct, single_agg, top_n=25):
    candidates = single_agg.head(top_n).copy()

    summaries = []
    days = []

    for _, row in candidates.iterrows():
        product_name = row["product"]
        signal_name = row["signal"]

        for invert in [False, True]:
            for entry_z in [0.75, 1.0, 1.5, 2.0]:
                summary, day_df = backtest_single_product_candidate(
                    cp=cp,
                    ct=ct,
                    product_name=product_name,
                    signal_name=signal_name,
                    entry_z=entry_z,
                    exit_z=0.0,
                    pos_limit=10,
                    invert=invert,
                    standardize=True,
                )

                if summary is None:
                    continue

                summary["category"] = category_name
                summary["source_score"] = row["single_score"]
                summaries.append(summary)

                day_df["category"] = category_name
                day_df["source_score"] = row["single_score"]
                days.append(day_df)

    summary_df = pd.DataFrame(summaries)
    day_df = pd.concat(days, ignore_index=True) if days else pd.DataFrame()

    if len(summary_df):
        summary_df = summary_df.sort_values("total_liquidated_pnl", ascending=False)

    return summary_df, day_df

In [7]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — RUN DEEP SCAN ACROSS ALL CATEGORIES
# ════════════════════════════════════════════════════════════════════════════

all_diag = []
all_trade_diag = []
all_corr = []
all_basket_raw = []
all_basket_agg = []
all_pair_raw = []
all_pair_agg = []
all_pca_raw = []
all_pca_agg = []
all_pca_explained = []
all_pca_loadings = []
all_single_raw = []
all_single_agg = []
all_leadlag_raw = []
all_leadlag_agg = []
all_backtest_summary = []
all_backtest_days = []

for category_name in CATEGORIES:
    print(f"\n===== {category_name} =====")

    products, cp, ct, wide = get_category_data(category_name)

    diag, trade_diag = scan_product_diagnostics(category_name, products, cp, ct)
    corr = scan_return_correlations(category_name, products, wide)
    basket_raw, basket_agg = scan_basket_invariants(category_name, products, wide)
    pair_raw, pair_agg = scan_pair_spreads(category_name, products, wide)
    pca_raw, pca_agg, pca_exp, pca_load = scan_pca_residuals(category_name, products, wide)
    single_raw, single_agg = scan_single_product_signals(category_name, products, cp, ct)
    leadlag_raw, leadlag_agg = scan_cross_product_leadlag(category_name, products, wide)

    # Conservative executable backtest only for top single-product signals.
    bt_summary, bt_days = backtest_top_single_candidates(
        category_name=category_name,
        cp=cp,
        ct=ct,
        single_agg=single_agg,
        top_n=TOP_N_BACKTEST_PER_CATEGORY,
    )

    all_diag.append(diag)
    all_trade_diag.append(trade_diag)
    all_corr.append(corr)
    all_basket_raw.append(basket_raw)
    all_basket_agg.append(basket_agg)
    all_pair_raw.append(pair_raw)
    all_pair_agg.append(pair_agg)
    all_pca_raw.append(pca_raw)
    all_pca_agg.append(pca_agg)
    all_pca_explained.append(pca_exp)
    all_pca_loadings.append(pca_load)
    all_single_raw.append(single_raw)
    all_single_agg.append(single_agg)
    all_leadlag_raw.append(leadlag_raw)
    all_leadlag_agg.append(leadlag_agg)
    all_backtest_summary.append(bt_summary)
    all_backtest_days.append(bt_days)

    print("Top single signals:")
    display(single_agg.head(5))

    print("Top executable backtests:")
    display(bt_summary.head(5))

diag_df = pd.concat(all_diag, ignore_index=True)
trade_diag_df = pd.concat(all_trade_diag, ignore_index=True)
corr_df = pd.concat(all_corr, ignore_index=True)
basket_raw_df = pd.concat(all_basket_raw, ignore_index=True)
basket_agg_df = pd.concat(all_basket_agg, ignore_index=True)
pair_raw_df = pd.concat(all_pair_raw, ignore_index=True)
pair_agg_df = pd.concat(all_pair_agg, ignore_index=True)
pca_raw_df = pd.concat(all_pca_raw, ignore_index=True)
pca_agg_df = pd.concat(all_pca_agg, ignore_index=True)
pca_explained_df = pd.concat(all_pca_explained, ignore_index=True)
pca_loadings_df = pd.concat(all_pca_loadings, ignore_index=True)
single_raw_df = pd.concat(all_single_raw, ignore_index=True)
single_agg_df = pd.concat(all_single_agg, ignore_index=True)
leadlag_raw_df = pd.concat(all_leadlag_raw, ignore_index=True)
leadlag_agg_df = pd.concat(all_leadlag_agg, ignore_index=True)
backtest_summary_df = pd.concat(all_backtest_summary, ignore_index=True)
backtest_days_df = pd.concat(all_backtest_days, ignore_index=True)

print("DONE.")


===== Galaxy Sounds =====
Top single signals:


,category,product,signal,future_h,days,avg_corr,min_corr,max_corr,avg_hit,avg_edge,min_edge,max_edge,avg_n,same_positive_corr,same_negative_corr,same_positive_edge,same_negative_edge,abs_avg_corr,single_score
447,Galaxy Sounds,GALAXY_SOUNDS_DARK_MATTER,meanrev_z_2500,250,3,0.243509,0.121636,0.390574,0.559248,24.487479,9.627878,42.094963,9251.0,True,False,True,False,0.243509,116.551755
367,Galaxy Sounds,GALAXY_SOUNDS_DARK_MATTER,breakout_z_2500,250,3,-0.243509,-0.390574,-0.121636,0.440752,-24.487479,-42.094963,-9.627878,9251.0,False,True,False,True,0.243509,116.551755
431,Galaxy Sounds,GALAXY_SOUNDS_DARK_MATTER,meanrev_z_1000,250,3,0.176530,0.074786,0.292279,0.541963,20.424441,4.825359,29.420218,9551.0,True,False,True,False,0.176530,90.395104
351,Galaxy Sounds,GALAXY_SOUNDS_DARK_MATTER,breakout_z_1000,250,3,-0.176530,-0.292279,-0.074786,0.458037,-20.424441,-29.420218,-4.825359,9551.0,False,True,False,True,0.176530,90.395104
366,Galaxy Sounds,GALAXY_SOUNDS_DARK_MATTER,breakout_z_2500,100,3,-0.170056,-0.265190,-0.095502,0.448213,-11.595539,-13.325231,-8.510690,9401.0,False,True,False,True,0.170056,70.178632


Top executable backtests:


,product,signal,entry_z,exit_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,category,source_score
87,GALAXY_SOUNDS_SOLAR_FLAMES,flow_imb_10000,2.0,0.0,True,14.0,14.0,4.666667,-9580.0,6954.0,2,1008,103,Galaxy Sounds,50.844979
127,GALAXY_SOUNDS_SOLAR_FLAMES,flow_imb_10000,2.0,0.0,True,14.0,14.0,4.666667,-9580.0,6954.0,2,1008,103,Galaxy Sounds,39.948610
79,GALAXY_SOUNDS_DARK_MATTER,meanrev_z_1000,2.0,0.0,True,-153.0,-288.0,-96.000000,-12436.0,16562.0,1,1450,149,Galaxy Sounds,52.194786
67,GALAXY_SOUNDS_DARK_MATTER,breakout_z_1000,2.0,0.0,False,-153.0,-288.0,-96.000000,-12436.0,16562.0,1,1450,149,Galaxy Sounds,52.194786
27,GALAXY_SOUNDS_DARK_MATTER,breakout_z_1000,2.0,0.0,False,-153.0,-288.0,-96.000000,-12436.0,16562.0,1,1450,149,Galaxy Sounds,90.395104



===== Sleeping Pods =====
Top single signals:


...


===== Snackpacks =====
Top single signals:


,category,product,signal,future_h,days,avg_corr,min_corr,max_corr,avg_hit,avg_edge,min_edge,max_edge,avg_n,same_positive_corr,same_negative_corr,same_positive_edge,same_negative_edge,abs_avg_corr,single_score
47,Snackpacks,SNACKPACK_CHOCOLATE,breakout_z_2500,250,3,-0.211786,-0.332336,-0.134306,0.443875,-15.460851,-24.788617,-7.322019,9251.0,False,True,False,True,0.211786,88.752808
127,Snackpacks,SNACKPACK_CHOCOLATE,meanrev_z_2500,250,3,0.211786,0.134306,0.332336,0.556125,15.460851,7.322019,24.788617,9251.0,True,False,True,False,0.211786,88.752808
1407,Snackpacks,SNACKPACK_VANILLA,meanrev_z_2500,250,3,0.198160,0.059699,0.347567,0.529771,11.442222,4.408713,20.756837,9251.0,True,False,True,False,0.198160,73.680241
1327,Snackpacks,SNACKPACK_VANILLA,breakout_z_2500,250,3,-0.198160,-0.347567,-0.059699,0.470229,-11.442222,-20.756837,-4.408713,9251.0,False,True,False,True,0.198160,73.680241
687,Snackpacks,SNACKPACK_RASPBERRY,breakout_z_2500,250,3,-0.163221,-0.275067,-0.086777,0.466540,-13.453050,-30.385364,-2.928927,9251.0,False,True,False,True,0.163221,70.758251


Top executable backtests:


,product,signal,entry_z,exit_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,category,source_score
42,SNACKPACK_RASPBERRY,meanrev_z_2500,1.5,0.0,False,-5156.0,-5326.0,-1775.333333,-6160.0,1264.0,1,2322,235,Snackpacks,70.758251
38,SNACKPACK_RASPBERRY,breakout_z_2500,1.5,0.0,True,-5156.0,-5326.0,-1775.333333,-6160.0,1264.0,1,2322,235,Snackpacks,70.758251
39,SNACKPACK_RASPBERRY,breakout_z_2500,2.0,0.0,True,-8864.0,-9034.0,-3011.333333,-6670.0,-980.0,0,1486,150,Snackpacks,70.758251
43,SNACKPACK_RASPBERRY,meanrev_z_2500,2.0,0.0,False,-8864.0,-9034.0,-3011.333333,-6670.0,-980.0,0,1486,150,Snackpacks,70.758251
37,SNACKPACK_RASPBERRY,breakout_z_2500,1.0,0.0,True,-9139.0,-9309.0,-3103.000000,-11660.0,2661.0,1,3484,351,Snackpacks,70.758251


DONE.


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — SAVE DEEP SCAN OUTPUTS
# ════════════════════════════════════════════════════════════════════════════

outputs = {
    "diagnostics": diag_df,
    "trade_diagnostics": trade_diag_df,
    "return_correlations": corr_df,
    "basket_raw": basket_raw_df,
    "basket_agg": basket_agg_df,
    "pair_raw": pair_raw_df,
    "pair_agg": pair_agg_df,
    "pca_raw": pca_raw_df,
    "pca_agg": pca_agg_df,
    "pca_explained": pca_explained_df,
    "pca_loadings": pca_loadings_df,
    "single_raw": single_raw_df,
    "single_agg": single_agg_df,
    "leadlag_raw": leadlag_raw_df,
    "leadlag_agg": leadlag_agg_df,
    "backtest_summary": backtest_summary_df,
    "backtest_days": backtest_days_df,
}

for name, df in outputs.items():
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved {name}: {path.resolve()} shape={df.shape}")

# Compact markdown report
report = []
report.append("# Round 5 Deep Scan Report\n")

def add_table(title, df, n=30):
    report.append(f"\n## {title}\n")
    if df is None or len(df) == 0:
        report.append("No rows.\n")
        return
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_table(
    "Top Executable Single-Product Backtests",
    backtest_summary_df.sort_values("total_liquidated_pnl", ascending=False),
    50,
)

add_table(
    "Top Single-Product Predictive Signals",
    single_agg_df.sort_values("single_score", ascending=False),
    50,
)

add_table(
    "Top Lead-Lag Signals",
    leadlag_agg_df.sort_values("leadlag_score", ascending=False),
    50,
)

add_table(
    "Top Pair-Spread Candidates",
    pair_agg_df.sort_values("pair_score", ascending=False),
    50,
)

add_table(
    "Top Basket / Integer Invariant Candidates",
    basket_agg_df.sort_values(["range_to_mean", "avg_range"], ascending=True),
    50,
)

add_table(
    "PCA Explained Variance by Category",
    pca_explained_df,
    50,
)

# Category-level summary
cat_summary = (
    backtest_summary_df
    .groupby("category")
    .agg(
        best_liquidated_pnl=("total_liquidated_pnl", "max"),
        median_liquidated_pnl=("total_liquidated_pnl", "median"),
        positive_backtests=("total_liquidated_pnl", lambda x: int((x > 0).sum())),
        tested_backtests=("total_liquidated_pnl", "count"),
    )
    .reset_index()
    .sort_values("best_liquidated_pnl", ascending=False)
)

add_table("Category Backtest Summary", cat_summary, 30)

report_path = OUT_DIR / "round5_deep_scan_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print(f"\nSaved report: {report_path.resolve()}")
display(cat_summary)

Saved diagnostics: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/diagnostics.csv shape=(150, 13)
Saved trade_diagnostics: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/trade_diagnostics.csv shape=(150, 6)
Saved return_correlations: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/return_correlations.csv shape=(800, 6)
Saved basket_raw: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/basket_raw.csv shape=(43080, 9)
Saved basket_agg: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/basket_agg.csv shape=(14360, 10)
Saved pair_raw: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/pair_raw.csv shape=(1500, 14)
Saved pair_agg: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/pair_agg.csv shape=(500, 15)
Saved pca_raw: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/pca_raw.csv shape=(750

,category,best_liquidated_pnl,median_liquidated_pnl,positive_backtests,tested_backtests
4,Pebbles,87837.0,-13969.0,83,200
3,Panels,31841.0,-16027.5,57,200
1,Organic Microchips,19917.0,-10565.0,54,200
5,Robots,13237.0,-9586.5,51,200
6,Sleeping Pods,12282.0,-14472.0,18,200
8,Translators,8703.0,-14151.0,40,200
2,Oxygen Shakes,2645.0,-33121.5,8,200
9,UV Visors,2564.0,-72817.5,2,200
0,Galaxy Sounds,14.0,-19830.0,2,200
7,Snackpacks,-5326.0,-98757.0,0,200


In [9]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — PRIORITY QUEUE
# ════════════════════════════════════════════════════════════════════════════

priority_rows = []

for category in CATEGORIES:
    bt = backtest_summary_df[backtest_summary_df["category"] == category].copy()
    single = single_agg_df[single_agg_df["category"] == category].copy()
    lead = leadlag_agg_df[leadlag_agg_df["category"] == category].copy()
    pair = pair_agg_df[pair_agg_df["category"] == category].copy()
    basket = basket_agg_df[basket_agg_df["category"] == category].copy()
    pca = pca_explained_df[pca_explained_df["category"] == category].copy()

    best_bt = bt.sort_values("total_liquidated_pnl", ascending=False).head(1)
    best_single = single.sort_values("single_score", ascending=False).head(1)
    best_lead = lead.sort_values("leadlag_score", ascending=False).head(1)
    best_pair = pair.sort_values("pair_score", ascending=False).head(1)
    best_basket = basket.sort_values(["range_to_mean", "avg_range"], ascending=True).head(1)

    row = {"category": category}

    if len(best_bt):
        r = best_bt.iloc[0]
        row["best_exec_pnl"] = r["total_liquidated_pnl"]
        row["best_exec_product"] = r["product"]
        row["best_exec_signal"] = r["signal"]
        row["best_exec_entry_z"] = r["entry_z"]
        row["best_exec_invert"] = r["invert"]

    if len(best_single):
        r = best_single.iloc[0]
        row["best_single_product"] = r["product"]
        row["best_single_signal"] = r["signal"]
        row["best_single_future_h"] = r["future_h"]
        row["best_single_score"] = r["single_score"]

    if len(best_lead):
        r = best_lead.iloc[0]
        row["best_leadlag"] = f"{r['leader']} -> {r['follower']}"
        row["best_leadlag_past_future"] = f"{r['past_h']}->{r['future_h']}"
        row["best_leadlag_score"] = r["leadlag_score"]

    if len(best_pair):
        r = best_pair.iloc[0]
        row["best_pair"] = f"{r['a']} / {r['b']}"
        row["best_pair_window"] = r["window"]
        row["best_pair_score"] = r["pair_score"]

    if len(best_basket):
        r = best_basket.iloc[0]
        row["best_basket_type"] = r["type"]
        row["best_basket_coefs"] = r["coefs"]
        row["best_basket_range_to_mean"] = r["range_to_mean"]
        row["best_basket_avg_range"] = r["avg_range"]

    if len(pca):
        row["avg_pc1"] = pca["pc1"].mean()

    priority_rows.append(row)

priority_df = pd.DataFrame(priority_rows)
priority_df = priority_df.sort_values("best_exec_pnl", ascending=False)

priority_path = OUT_DIR / "priority_queue.csv"
priority_df.to_csv(priority_path, index=False)

display(priority_df)
print(f"Saved priority queue: {priority_path.resolve()}")

,category,best_exec_pnl,best_exec_product,best_exec_signal,best_exec_entry_z,best_exec_invert,best_single_product,best_single_signal,best_single_future_h,best_single_score,...,best_leadlag_past_future,best_leadlag_score,best_pair,best_pair_window,best_pair_score,best_basket_type,best_basket_coefs,best_basket_range_to_mean,best_basket_avg_range,avg_pc1
3,Pebbles,87837.0,PEBBLES_XL,breakout_z_250,2.00,True,PEBBLES_XL,breakout_z_250,250,133.144493,...,250->250,126.444977,PEBBLES_S / PEBBLES_XL,1000,15.985620,equal_sum,"[1, 1, 1, 1, 1]",0.000693,34.666667,0.580961
7,Panels,31841.0,PANEL_1X4,breakout_z_1000,1.50,False,PANEL_1X4,breakout_z_1000,250,77.028324,...,250->250,98.404793,PANEL_2X2 / PANEL_4X4,250,19.287971,integer_combo,"[2, 2, 1, 1, 2]",0.054078,4186.666667,0.534798
2,Organic Microchips,19917.0,MICROCHIP_TRIANGLE,meanrev_z_1000,1.50,True,MICROCHIP_TRIANGLE,meanrev_z_2500,250,180.657278,...,250->250,114.605779,MICROCHIP_CIRCLE / MICROCHIP_OVAL,250,19.315328,integer_combo,"[2, 1, 1, 0, 1]",0.071560,3570.166667,0.589043
4,Robots,13237.0,ROBOT_LAUNDRY,mom_ret_250,1.50,True,ROBOT_LAUNDRY,meanrev_z_2500,250,91.100174,...,250->250,110.570989,ROBOT_VACUUMING / ROBOT_MOPPING,250,23.953604,integer_combo,"[2, 2, 2, 2, 1]",0.043655,3881.666667,0.549371
1,Sleeping Pods,12282.0,SLEEP_POD_LAMB_WOOL,flow_imb_10000,0.75,False,SLEEP_POD_NYLON,breakout_z_2500,250,137.756474,...,250->250,77.283509,SLEEP_POD_NYLON / SLEEP_POD_COTTON,500,20.350444,integer_combo,"[2, 1, 1, 2, 1]",0.071056,5410.000000,0.624620
6,Translators,8703.0,TRANSLATOR_SPACE_GRAY,breakout_z_2500,1.50,False,TRANSLATOR_GRAPHITE_MIST,breakout_z_2500,250,88.330373,...,250->250,86.632899,TRANSLATOR_ASTRO_BLACK / TRANSLATOR_ECLIPSE_CH...,100,18.240972,integer_combo,"[2, 2, 1, 1, 2]",0.062072,4919.166667,0.578554
8,Oxygen Shakes,2645.0,OXYGEN_SHAKE_GARLIC,meanrev_z_500,2.00,True,OXYGEN_SHAKE_GARLIC,meanrev_z_500,250,57.598818,...,250->250,91.115947,OXYGEN_SHAKE_CHOCOLATE / OXYGEN_SHAKE_GARLIC,100,18.606735,integer_combo,"[1, 2, 1, 1, 2]",0.059924,4302.000000,0.486169
5,UV Visors,2564.0,UV_VISOR_RED,meanrev_z_2500,2.00,True,UV_VISOR_RED,breakout_z_2500,250,83.479316,...,250->250,128.489124,UV_VISOR_AMBER / UV_VISOR_MAGENTA,100,19.131247,integer_combo,"[1, 2, 1, 2, 2]",0.049429,4031.333333,0.450487
0,Galaxy Sounds,14.0,GALAXY_SOUNDS_SOLAR_FLAMES,flow_imb_10000,2.00,True,GALAXY_SOUNDS_DARK_MATTER,meanrev_z_2500,250,116.551755,...,250->250,97.206009,GALAXY_SOUNDS_DARK_MATTER / GALAXY_SOUNDS_SOLA...,250,17.903625,integer_combo,"[2, 1, 1, 2, 2]",0.061052,5235.000000,0.491066
9,Snackpacks,-5326.0,SNACKPACK_RASPBERRY,meanrev_z_2500,1.50,False,SNACKPACK_CHOCOLATE,breakout_z_2500,250,88.752808,...,250->250,58.318920,SNACKPACK_CHOCOLATE / SNACKPACK_VANILLA,1000,121.607648,integer_combo,"[2, 2, 2, 1, 2]",0.009719,872.166667,0.525857


Saved priority queue: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_deep_scan/priority_queue.csv
